In [1]:
import pandas as pd

listings = pd.read_csv('data/listings.csv')

listings.shape

(4574, 84)

## Count per group: value_counts

describe() 4 different types of rules. We only looked at one of these rules. Today we will use value counts.

In [ ]:
listings['room_type'].value_counts() # Count the number of listings for each room type, sorted by counts in descending order. The sum of the counts should equal the number of rows in the DataFrame (4574).

room_type
Entire home/apt    3760
Private room        787
Shared room          24
Hotel room            3
Name: count, dtype: int64

In [ ]:
listings['room_type'].value_counts(normalize=True).round(3) # Count the number of listings for each room type, sorted by counts in descending order. The sum of the counts should equal 1.0 (the normalization).

room_type
Entire home/apt    0.822
Private room       0.172
Shared room        0.005
Hotel room         0.001
Name: proportion, dtype: float64

In [4]:
listings['neighbourhood_cleansed'].value_counts() # Count the number of listings for each neighbourhood, sorted by counts in descending order. The sum of the counts should equal the number of rows in the DataFrame (4574).

neighbourhood_cleansed
Södermalms                1145
Norrmalms                  469
Hägersten-Liljeholmens     464
Enskede-Årsta-Vantörs      445
Kungsholmens               420
Östermalms                 297
Bromma                     281
Skarpnäcks                 221
Älvsjö                     214
Farsta                     177
Hässelby-Vällingby         148
Rinkeby-Tensta             108
Skärholmens                 96
Spånga-Tensta               89
Name: count, dtype: int64

In [5]:
listings['bedrooms'].value_counts(dropna=False).sort_index() # Count the number of listings for each number of bedrooms, including NaN values, sorted by the number of bedrooms in ascending order. The sum of the counts should equal the number of rows in the DataFrame (4574).

bedrooms
0.0      257
1.0     2490
2.0      739
3.0      588
4.0      351
5.0      116
6.0       23
7.0        3
8.0        1
12.0       4
NaN        2
Name: count, dtype: int64

## Group and calculate: groupby

1. Split
2. Calculate
3. Combine

In [6]:
listings.groupby('room_type')['accommodates'].mean().round(2) # Calculate the average number of accommodates for each room type, rounded to one decimal place.

room_type
Entire home/apt    3.97
Hotel room         1.33
Private room       1.93
Shared room        4.46
Name: accommodates, dtype: float64

In [7]:
listings.groupby('room_type')['accommodates'].agg(['count', 'mean', 'median', 'max']).round(2) # Calculate the count, mean, median, and maximum number of accommodates for each room type, rounded to two decimal places.

,count,mean,median,max
room_type,,,,
Entire home/apt,3760,3.97,4.0,16
Hotel room,3,1.33,1.0,2
Private room,787,1.93,2.0,16
Shared room,24,4.46,2.5,14


In [ ]:
listings.groupby('room_type')[['accommodates', 'bedrooms', 'beds']].mean().round(2) # Calculate the average number of accommodates, bedrooms, and beds (3 columns in this [[list]] that we created right here) for each room type, rounded to two decimal places.

,accommodates,bedrooms,beds
room_type,,,
Entire home/apt,3.97,1.87,2.48
Hotel room,1.33,1.00,1.00
Private room,1.93,1.11,1.48
Shared room,4.46,0.79,4.58


In [ ]:
listings.groupby('room_type')['price'].mean() # Produces an error because the price column is a string. We need to convert it to a numeric type first.TypeError: dtype 'str' does not support operation 'mean'

In [12]:
summary = listings.groupby('room_type').agg(
    n_listings=('id', 'count'),
    avg_guests=('accommodates', 'mean'),
    median_reviews=('number_of_reviews', 'median')
).round(2) # Calculate the count of listings, average number of guests, and median number of reviews for each room type, rounded to two decimal places.
summary

,n_listings,avg_guests,median_reviews
room_type,,,
Entire home/apt,3760,3.97,8.0
Hotel room,3,1.33,56.0
Private room,787,1.93,12.0
Shared room,24,4.46,81.5


In [ ]:
(listings.groupby('neighbourhood_cleansed')['review_scores_rating'].agg(['count', 'mean', 'median']).round(2).sort_values('mean')) # Calculate the count, mean, and median of review scores for each neighbourhood, rounded to two decimal places, and sorted by the mean (column) review score in ascending order.

## Two categories at once: crosstab

In [3]:
listings.groupby(['neighbourhood_cleansed', 'room_type']).size().head(8) # Count the number of listings for each combination of neighbourhood and room type, sorted by counts in descending order. The sum of the counts should equal the number of rows in the DataFrame (4574).

neighbourhood_cleansed  room_type      
Bromma                  Entire home/apt    259
                        Private room        22
Enskede-Årsta-Vantörs   Entire home/apt    362
                        Private room        83
Farsta                  Entire home/apt    145
                        Private room        32
Hägersten-Liljeholmens  Entire home/apt    394
                        Private room        70
dtype: int64

In [4]:
pd.crosstab(listings['neighbourhood_cleansed'], listings['room_type'], margins=True) # Create a cross-tabulation of the number of listings for each combination of neighbourhood and room type, including row and column totals.

room_type,Entire home/apt,Hotel room,Private room,Shared room,All
neighbourhood_cleansed,,,,,
Bromma,259,0,22,0,281
Enskede-Årsta-Vantörs,362,0,83,0,445
Farsta,145,0,32,0,177
Hägersten-Liljeholmens,394,0,70,0,464
Hässelby-Vällingby,118,0,30,0,148
Kungsholmens,332,0,80,8,420
Norrmalms,373,3,86,7,469
Rinkeby-Tensta,74,0,33,1,108
Skarpnäcks,167,0,54,0,221


In [6]:
(pd.crosstab(listings['neighbourhood_cleansed'], listings['room_type'], margins=True, normalize='index') * 100).round(1) # Create a cross-tabulation of the number of listings for each combination of neighbourhood and room type, including row and column totals, and normalize the counts by the row totals to get percentages rounded to three decimal places.

room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Bromma,92.2,0.0,7.8,0.0
Enskede-Årsta-Vantörs,81.3,0.0,18.7,0.0
Farsta,81.9,0.0,18.1,0.0
Hägersten-Liljeholmens,84.9,0.0,15.1,0.0
Hässelby-Vällingby,79.7,0.0,20.3,0.0
Kungsholmens,79.0,0.0,19.0,1.9
Norrmalms,79.5,0.6,18.3,1.5
Rinkeby-Tensta,68.5,0.0,30.6,0.9
Skarpnäcks,75.6,0.0,24.4,0.0


## Create new columns: np.where

In [7]:
listings['for_groups'] = listings['accommodates'] >= 4 # Create a new column called 'for_groups' that indicates whether the listing can accommodate 4 or more guests (True) or not (False).
listings[['accommodates', 'for_groups']].head(6) # Display the first 6 rows of the 'accommodates' and 'for_groups' columns to verify that the new column has been created correctly.

,accommodates,for_groups
0,2,False
1,1,False
2,1,False
3,4,True
4,5,True
5,2,False


In [8]:
listings.shape

(4574, 85)

*Copy-on-Write*

In [9]:
guests = listings['accommodates']

guests.iloc[0] = 99

print(guests.iloc[0])
print(listings.loc[0, 'accommodates'])

99
2


In [10]:
import numpy as np

listings['size_type'] = np.where(listings['accommodates'] >= 4, 'group', 'small')

listings['size_type'].value_counts() # Count the number of listings for each size type, sorted by counts in descending order. The sum of the counts should equal the number of rows in the DataFrame (4574).

size_type
small    2405
group    2169
Name: count, dtype: int64

In [15]:
listings['size_three'] = np.where(listings['accommodates'] >= 2, 'small',
np.where(listings['accommodates'] <= 4, 'medium', 'large')) # Create a new column called 'size_three' that indicates whether the listing can accommodate 4 or more guests (group) or not (small).
listings['size_three'].value_counts() # Count the number of listings for each size type, sorted by counts in descending order. The sum of the counts should equal the number of rows in the DataFrame (4574).

size_three
small     4182
medium     392
Name: count, dtype: int64

Turn numbers intop groups: cut and qcut

In [ ]:
pd.cut(listings['accommodates'], bins=[0, 2, 4, 8, 16]).value_counts().sort_index() # Count the number of listings for each range of accommodates, sorted by the range in ascending order. The sum of the counts should equal the number of rows in the DataFrame (4574). Right boarder is not included in the range while the left boarder is included. For example, the range (0, 2] includes all values greater than 0 and less than or equal to 2.

accommodates
(0, 2]     1882
(2, 4]     1478
(4, 8]     1117
(8, 16]      97
Name: count, dtype: int64

In [ ]:
print(pd.cut(listings['accommodates'], bins=[1, 2, 4, 8, 16]).isna().sum()) # Count the number of listings that fall outside the specified ranges (i.e., have NaN values) in the 'accommodates' column. The sum of the counts should equal the number of rows in the DataFrame (4574) minus the number of listings that fall within the specified ranges. 392 listings have accommodates = 0, which is outside the specified ranges, so the output should be 392 ("they do not get a group").

392


In [26]:
listings['size_group'] = pd.cut(listings['accommodates'], bins=[0, 2, 4, 8, 16], labels=['1-2 small', '3-4 medium', '5-8 large', '9-16 extra large']) # Create a new column called 'size_group' that categorizes the listings into size groups based on the number of accommodates. The size groups are defined by the specified bins and labels.
print(listings['size_group'].isna().sum()) # Count the number of listings that fall outside the specified ranges (i.e., have NaN values) in the 'size_group' column. The sum of the counts should equal the number of rows in the DataFrame (4574) minus the number of listings that fall within the specified ranges. 392 listings have accommodates = 0, which is outside the specified ranges, so the output should be 392 ("they do not get a group").
listings['size_group'].value_counts().sort_index() 

0


size_group
1-2 small           1882
3-4 medium          1478
5-8 large           1117
9-16 extra large      97
Name: count, dtype: int64

In [28]:
pd.qcut(listings['number_of_reviews'], 4).value_counts().sort_index() # Count the number of listings for each quartile of the 'number_of_reviews' column, sorted by the quartile ranges in ascending order. The sum of the counts should equal the number of rows in the DataFrame (4574). Right boarder is not included in the range while the left boarder is included. For example, the range (0, 2] includes all values greater than 0 and less than or equal to 2.

number_of_reviews
(-0.001, 2.0]     1227
(2.0, 9.0]        1083
(9.0, 29.0]       1121
(29.0, 1293.0]    1143
Name: count, dtype: int64

In [29]:
listings['number_of_reviews'].describe() # Get a summary of the 'number_of_reviews' column, including count, mean, standard deviation, minimum, 25th percentile, median (50th percentile), 75th percentile, and maximum values.

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64

In [30]:
pd.qcut[listings['availability_365']], 4

TypeError: 'function' object is not subscriptable

In [31]:
print((listings['availability_365'] == 0).sum()) # Count the number of listings that have an availability of 0 days in the 'availability_365' column. The sum of the counts should equal the number of rows in the DataFrame (4574) minus the number of listings that have an availability greater than 0.

1494


In [33]:
pd.qcut(listings['availability_365'], 4, duplicates='drop').value_counts().sort_index() # Count the number of listings for each quartile of the 'availability_365' column, sorted by the quartile ranges in ascending order. The sum of the counts should equal the number of rows in the DataFrame (4574). Right boarder is not included in the range while the left boarder is included. For example, the range (0, 2] includes all values greater than 0 and less than or equal to 2. The duplicates='drop' parameter is used to drop any duplicate bin edges that may occur when there are fewer unique values than the specified number of quantiles.

availability_365
(-0.001, 84.0]    2302
(84.0, 277.0]     1131
(277.0, 365.0]    1141
Name: count, dtype: int64

## Put it together

**Which neighbourhoods have the highest and the lowest share of listings for groups, with room for four or more guests? And how many listings is each share based on?**

1. Group by neighbourhood
2. Calculate
3. Sort by this share

In [34]:
answer = (
    listings.groupby('neighbourhood_cleansed')
    .agg(n_listings=('id', 'count'), share_for_groups=('for_groups', 'mean'))
    .sort_values('share_for_groups', ascending=False)
    .round(3)
)

answer

,n_listings,share_for_groups
neighbourhood_cleansed,,
Bromma,281,0.712
Farsta,177,0.644
Älvsjö,214,0.612
Hässelby-Vällingby,148,0.608
Skarpnäcks,221,0.538
Enskede-Årsta-Vantörs,445,0.526
Hägersten-Liljeholmens,464,0.491
Spånga-Tensta,89,0.461
Kungsholmens,420,0.426
